# Step 0: Setup and imports

In [1]:
import pandas as pd, numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
# pip install solas-ai-disparity
# from solas_ai.disparity import DisparityTest
import solas_disparity as sd
# Load and clean COMPAS data (Lecture 01 pipeline)
# df = clean dataframe from lecture 02
# print(f"N = {len(df):,}")
# print(df[['race','sex','high_risk','two_year_recid']].head())


ModuleNotFoundError: No module named 'solas_disparity'

In [12]:
# !pip install solas-ai

!pip install solas-ai-disparity==0.5.1

ERROR: Could not find a version that satisfies the requirement solas-ai-disparity==0.5.1 (from versions: none)
ERROR: No matching distribution found for solas-ai-disparity==0.5.1


# Step 1: AIR and Marginal Effect by race

In [ ]:
def selection_rate(df, group_col, outcome_col, ref_group):
  """Selection rates, AIR, and ME relative to reference group"""
  rates = (df.groupby(group_col)[outcome_col].mean().rename('selection_rate').reset_index())
  ref_rate = rates.loc[rates[group_col]==ref_group, 'selection_rate'].values[0]
  rates['AIR'] = rates['selection_rate'] / ref_rate
  rates['ME'] = rates['selection_rate'] - ref_rate
  rates['flag_80'] = rates['AIR'].apply(lambda x: '*** BELOW 0.80' if x < 0.80 else '')
  return rates
sir = selection_rate(df, 'race', 'high_risk', ref_group='Caucasian')
print(sir.sort_values('AIR').to_string(index=False))

In [ ]:
# Two-proportion z-test: Black vs. White
groups = ['African-American', 'Caucasian']
ns = df[df['race'].isin(groups)].groupby('race')['high_risk'].count()
events = df[df['race'].isin(groups)].groupby('race')['high_risk'].sum()
stat, pval = proportions_ztest(events[groups].values, ns[groups].values)
print(f"\nAIR test: z = {stat:.3f}, p = {pval:.4f}")

# Step 2: Error-Rate Disparity Analysis

In [ ]:
# FPR and FNR by race
def error_rates(df, group_col, pred_col, outcome_col):
  results = []
  for grp, g in df.groupby(group_col):
  tp = ((g[pred_col]==1) & (g[outcome_col]==1)).sum()
  tn = ((g[pred_col]==0) & (g[outcome_col]==0)).sum()
  fp = ((g[pred_col]==1) & (g[outcome_col]==0)).sum()
  fn = ((g[pred_col]==0) & (g[outcome_col]==1)).sum()
  results.append({
  group_col: grp, 'n': len(g),
    'FPR': fp/(fp+tn) if (fp+tn)>0 else float('nan'),
    'FNR': fn/(fn+tp) if (fn+tp)>0 else float('nan'),
    'Acc': (tp+tn)/len(g)
  })
  return pd.DataFrame(results)
er = error_rates(df, ’race’, ’high_risk’, ’two_year_recid’)
print(er.sort_values(’FPR’, ascending=False).to_string(index=False))
# Highlight Black vs. White disparity
for grp in ['African-American', 'Caucasian']:
row = er.loc[er['race'] == grp]
print(f"{grp}: FPR={row['FPR'].values[0]:.3f} ", f"FNR={row['FNR'].values[0]:.3f}")

# Step 3: Standardized Mean Difference

In [ ]:
# SMD on continuous COMPAS decile score
def smd(df, group_col, score_col, ref_group):
  """Cohen’s d vs. reference group"""
  ref = df.loc[df[group_col]==ref_group, score_col]
  results = []
  for grp, g in df.groupby(group_col):
    if grp == ref_group:
      continue
  sc = g[score_col]
  pooled = np.sqrt((ref.var() + sc.var()) / 2)
  d = ((sc.mean() - ref.mean()) / pooled if pooled > 0 else 0)
  mag = ('small' if abs(d) < 0.2 else 'medium' if abs(d) < 0.5 else 'large' if abs(d) < 0.8 else 'very large')
  results.append({group_col: grp,
                  'mean_score': round(sc.mean(), 3),
                  'SMD': round(d, 3),
                  'magnitude': mag})
  return pd.DataFrame(results)
smd_tbl = smd(df, 'race', 'decile_score', ref_group='Caucasian')
print(smd_tbl.sort_values('SMD', ascending=False).to_string(index=False))

# Step 4: Intersectional Subgroup Analysis

In [ ]:
# Intersectional analysis -- race x sex
df['subgroup'] = df['race'] + ' / ' + df['sex']
# Keep subgroups with n >= 30
counts = df['subgroup'].value_counts()
valid_sg = counts[counts >= 30].index
df_sub = df[df['subgroup'].isin(valid_sg)].copy()
sub_rates = (df_sub.groupby('subgroup')['high_risk'].agg(['mean','count']).rename(columns={'mean':'selection_rate','count':'n'}).reset_index())
ref_rate = sub_rates.loc[sub_rates['subgroup']=='Caucasian / Male','selection_rate'].values[0]
sub_rates['AIR'] = sub_rates['selection_rate'] / ref_rate
sub_rates['flag'] = sub_rates['AIR'].apply(lambda x: '*** BELOW 0.80' if x < 0.80 else '')
print(sub_rates.sort_values('AIR').to_string(index=False))
worst = sub_rates.loc[sub_rates['AIR'].idxmin()]
print(f"\nWorst: {worst['subgroup']}, AIR={worst['AIR']:.3f} and "
f"n={worst['n']}")